In [0]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession
import boto3
from datetime import *
import utils
from utils.constants import *
import sys
import os

sys.path.insert(0,'/Workspace/Repos/mushroomred933@gmail.com/Banking-data-platform-prod/')

In [0]:
def connect_to_s3(spark):
    spark.conf.set("fs.s3a.access.key",AWS_ACCESS_KEY_ID )
    spark.conf.set("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    spark.conf.set("fs.s3a.endpoint", "s3.amazonaws.com")
print("connected to s3")

In [0]:
def read_bronze_parquet(spark):
    path = (f"s3://banking-data-platform/bronze_parquet/cards/part-00000-271396aa-a5d4-4def-ab6f-e7adba674b9f-c000.snappy.parquet")
    df = spark.read.format("parquet")\
        .option("header", True)\
        .option("inferSchema", True)\
        .load(path)
    return df
df = read_bronze_parquet(spark)


        
def silver_transformation(df):
    df_silver = df\
            .withColumn(("cvv"),lpad(col("cvv"),3,"0"))\
            .withColumn("is_valid_cvv",
                 when((col("card_brand") == "Visa") & (col("card_number").startswith ("4")),"Is Valid")\
                .when((col("card_brand") == "Mastercard") & (col("card_number").startswith("5")),"Is Valid")\
                .when((col("card_brand") == "Discover") & (col("card_number").startswith("6")),"Is Valid")\
                .when((col("card_brand") == "Amex") & (col("card_number").startswith("3")),"Is valid")\
                .otherwise("Not valid")
            )\
            .withColumn("card_number_masked" ,concat(lit("XXXX-XXXX-XXXX-"),substring(col("card_number"),-4, 4) ) )\
            .orderBy("id")\
            .drop (col("card_number"))
    return df_silver
df_silver = silver_transformation(df)
#df_silver.display()
cols = df_silver.columns

cols.remove("card_number_masked")
cols.remove("id")
cols.remove("client_id")
cols.remove("card_type")
cols.remove("card_brand")
cols.remove("cvv")
cols.remove("expires")
new_order = ["id","client_id","card_brand","card_type","card_number_masked","cvv","expires"]+cols
df_silver = df_silver.select(*new_order)
#df_silver.display()

def write_silver(df_silver):
    path = f"s3://banking-data-platform/silver/cards/"
    df_silver.write.mode("overwrite").format("parquet").save(path)
    print("silver written")
write_silver(df_silver)
df_silver.display(10)
        
            
                   



    
